# FD-LEX Scriptoria: keşifsel veri analizi (Aşama 1 denetim + Aşama 2 ön fizibilite)

Veri: FD-LEX (Forschungsdatenbank Lernertexte), Scriptoria korpusu, DOI 10.18716/FD-LEX/861; 8 PDF + `Datentabelle.xlsx`.
Bu notebook yalnızca **okur ve ölçer**: ham veri değişmez, sayfa görüntüleri diske yazılmaz (örnek sayfalar bellekte işlenir),
model eğitilmez. Çizim modülüne ve Malaysian çalışmasına dokunulmaz. Bulgu özeti: `docs/fd_lex_dataset_analysis.md`.

Etiketler: **doğrulandı** (dosyadan ölçüldü), **yayında** (web kaynağında yazıyor, dosyadan doğrulanamadı), **bilinmiyor**.
Gizlilik: kapak sayfasında PDF'i indiren kullanıcının adı yazıyor; bu notebook ve manifest o satırı **saklamaz**.

In [ ]:
import hashlib, io, random, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pymupdf
from PIL import Image
from scipy import ndimage as ndi, stats
warnings.filterwarnings("ignore")
pymupdf.TOOLS.mupdf_display_errors(False)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
FD = ROOT / "data" / "handwriting" / "fd_lex"
RAW, META, MANIFESTS = FD / "raw", FD / "metadata", FD / "manifests"
MANIFESTS.mkdir(parents=True, exist_ok=True)
SEED = 42
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("repo:", ROOT.name, "| PDF:", len(list(RAW.glob("*.pdf"))), "| Excel:", [p.name for p in META.glob("*")], "| pymupdf", pymupdf.__version__)

## 1. PDF dosyaları ve kapak sayfaları (Aşama 1)

In [ ]:
pdfs = sorted(RAW.glob("*.pdf"))
inv, covers = [], {}
for p in pdfs:
    d = pymupdf.open(p); t = d[0].get_text()
    # kullanıcı adı satırını sakla(ma)
    t_clean = "\n".join(l for l in t.splitlines() if not l.startswith("Benutzer"))
    covers[p.name] = t_clean
    m = re.search(r"Datei: (\d+) / (\d+)", t); h = re.search(r"Hits: (\d+)", t)
    inv.append(dict(pdf=p.name, sayfa=d.page_count, mb=round(p.stat().st_size / 1e6), uretici=d.metadata.get("producer"),
                    datei=f"{m.group(1)}/{m.group(2)}" if m else None, hits=int(h.group(1)) if h else None, kapak=t.startswith("DECKBLATT")))
inv = pd.DataFrame(inv); print(inv.to_string(index=False))
print("\nToplam sayfa:", inv.sayfa.sum(), "| kapak sayfası:", int(inv.kapak.sum()), "| kapak hariç:", inv.sayfa.sum() - inv.kapak.sum())
print("Kapakta Hits değerleri:", inv.hits.unique().tolist(), "(toplam yazı sayısı; her dosyada aynı)")
same = len({re.sub(r"Datei: \d+ / \d+", "", c) for c in covers.values()}) == 1
print("Kapak metinleri (Datei numarası ve kullanıcı satırı hariç) birbirinin aynı mı:", same)
print("\nKapak metni (kullanıcı satırı çıkarıldı):\n" + covers[pdfs[0].name][:700])
print("\nKapakta filtre/seçim ölçütü satırı var mı:", any(k in covers[pdfs[0].name] for k in ["Filter", "Klasse", "Gruppe", "Schulform", "Suche"]),
      "→ kapak yalnızca korpus adı, dosya sırası, toplam kayıt sayısı (Hits) ve atıf bilgisi veriyor.")

## 2. Excel metadata (`Datentabelle.xlsx`)

In [ ]:
xl = pd.read_excel(META / "Datentabelle.xlsx", sheet_name=None)
print("Sayfalar:", {k: v.shape for k, v in xl.items()})
meta = xl["Selektion_Kopfdaten"].copy()
print(meta.columns.tolist())
print("Satır:", len(meta), "| benzersiz Code:", meta.Code.nunique(), "| Code uzunlukları:", meta.Code.str.len().value_counts().to_dict())
print("Eksik değerler:", {k: int(v) for k, v in meta.isna().sum().items() if v})
print("Beklenen yazı sayısı = öğrenci x 6 =", meta.Code.nunique() * 6, "(kapaktaki Hits ile aynı mı:", meta.Code.nunique() * 6 == inv.hits.iloc[0], ")")

## 3. Sayfa manifesti: bütün sayfaların metin katmanı, görüntü başlığı ve görüntü parmak izi
Sayfa görüntüleri diske yazılmaz. Ham akışın SHA-1 özeti (birebir kopya için) ve JPEG ise 24x32 gri küçük resim (yaklaşık kopya için, bellekte) hesaplanır.

In [ ]:
def thumb_from_jpeg(raw, size=(24, 32)):
    im = Image.open(io.BytesIO(raw)); im.draft("L", (im.width // 8, im.height // 8))
    a = np.asarray(im.convert("L").resize(size, Image.BILINEAR), dtype=np.float32)
    return a

CODE_RE = re.compile(r"^[0-9A-Za-zÄÖÜäöü]{6,9}$")
rows, thumbs = [], {}
for p in pdfs:
    d = pymupdf.open(p)
    for i, pg in enumerate(d):
        t = pg.get_text(); lines = [l.strip() for l in t.splitlines() if l.strip()]
        r = dict(pdf=p.name, seite=i + 1, is_cover=t.startswith("DECKBLATT"), rotation=pg.rotation,
                 page_w_pt=round(pg.rect.width, 1), page_h_pt=round(pg.rect.height, 1), text_len=len(t))
        if not r["is_cover"]:
            foot = next((k for k, l in enumerate(lines) if l.startswith("©")), None)
            before = lines[:foot] if foot is not None else lines
            after = lines[foot + 1:] if foot is not None else []
            cand = [l for l in before if CODE_RE.match(l) and l != "XXX"]
            r["code_text"] = cand[-1] if cand else None
            neu = [l[4:].strip() for l in after if l.startswith("neu:")]
            r["code_neu"] = neu[0] if neu else None
            r["n_xxx"] = sum(1 for l in before if l == "XXX")
            r["text_extra_lines"] = max(0, len(before) - 1 - r["n_xxx"])
        imgs = pg.get_images(full=True); r["n_img"] = len(imgs)
        if imgs:
            x = imgs[0]; xref = x[0]
            r.update(img_w=x[2], img_h=x[3], bpc=x[4], cs=x[5])
            r["filter"] = d.xref_get_key(xref, "Filter")[1]
            raw = d.xref_stream_raw(xref); r["img_bytes"] = len(raw); r["img_sha1"] = hashlib.sha1(raw).hexdigest()
            if "DCT" in str(r["filter"]) and not r["is_cover"]:
                try: thumbs[(p.name, i + 1)] = thumb_from_jpeg(raw)
                except Exception: pass
        rows.append(r)
pages = pd.DataFrame(rows)
print("Sayfa:", len(pages), "| kapak:", int(pages.is_cover.sum()), "| küçük resmi çıkarılan:", len(thumbs))
print("Görüntü filtresi:", pages[~pages.is_cover]["filter"].str.replace("/", "").value_counts().to_dict())

## 4. Öğrenci kodları: metin katmanından çıkarma ve Excel ile eşleştirme
Kod, sayfa altbilgisinin ("©") hemen üstündeki satırdır. Birkaç sayfada kod bozuk (örn. 7 karakter) ve "neu:" düzeltme satırı var.

In [ ]:
codes = set(meta.Code); upper = {c.upper(): c for c in codes}
def resolve(r):
    for name, c in [("metin", r.code_text), ("neu", r.code_neu), ("neu+0", "0" + str(r.code_neu)), ("neu+1", "1" + str(r.code_neu))]:
        if isinstance(c, str):
            if c in codes: return c, name
            if c.upper() in upper: return upper[c.upper()], name + "_buyuk"
    return None, None
body = pages[~pages.is_cover].copy()
res = body.apply(resolve, axis=1, result_type="expand"); body["code"], body["code_yontem"] = res[0], res[1]
print("Kod çözümü yöntemi:", body.code_yontem.fillna("ÇÖZÜLEMEDİ").value_counts().to_dict())
# metin katmanında kod yok / Excel'de yok: komşu sayfalardan çıkarım (öğrenci blokları ardışık)
idx = body.index.tolist(); body["code_cikarim"] = False
for _ in range(2):
    for k, i in enumerate(idx):
        if body.at[i, "code"] is None and 0 < k < len(idx) - 1:
            a, b = body.at[idx[k - 1], "code"], body.at[idx[k + 1], "code"]
            if a is not None and a == b and body.at[idx[k - 1], "pdf"] == body.at[i, "pdf"] == body.at[idx[k + 1], "pdf"]:
                body.at[i, "code"], body.at[i, "code_yontem"], body.at[i, "code_cikarim"] = a, "komsu_cikarim", True
print("Komşudan çıkarılan:", int(body.code_cikarim.sum()), "| hâlâ eşleşmeyen sayfa:", int(body.code.isna().sum()))
print("Eşleşmeyen sayfalardaki ham kodlar:", body[body.code.isna()][["pdf", "seite", "code_text", "code_neu"]].head(10).to_dict("records"))
print("\nSayfa düzeyinde eşleşme oranı: %.2f%%" % (100 * body.code.notna().mean()))
per = body.dropna(subset=["code"]).groupby("code").size()
print("Excel'deki öğrenci:", len(codes), "| sayfası bulunan:", per.index.nunique(), "| sayfası bulunmayan:", sorted(codes - set(per.index))[:10], len(codes - set(per.index)))
print("PDF'te kodu olup Excel'de olmayan (ham metin kodu):", sorted(set(body.code_text.dropna()) - codes - {c for c in body.code_neu.dropna()})[:15])
print("Öğrenci başına sayfa dağılımı:", per.value_counts().sort_index().to_dict())
print("Öğrenci eşleşme oranı: %.2f%%" % (100 * per.index.nunique() / len(codes)))

### Çok sayfalı yazılar, kopyalar ve PDF'ler arası örtüşme

In [ ]:
body = body.reset_index(drop=True)
body["blok"] = (body.code != body.code.shift()).cumsum()
blocks = body.dropna(subset=["code"]).groupby("code").blok.nunique()
print("Kodu birden çok ayrı blokta geçen öğrenci:", int((blocks > 1).sum()))
pdf_per_code = body.dropna(subset=["code"]).groupby("code").pdf.nunique()
print("Birden çok PDF'te görünen öğrenci (PDF'ler arası örtüşme):", int((pdf_per_code > 1).sum()))
extra = per[per != 6]
print("6 sayfa olmayan öğrenci:", len(extra), "| fazladan sayfa toplamı:", int((per - 6).clip(lower=0).sum()), "| eksik sayfa:", int((6 - per).clip(lower=0).sum()))
print(extra.value_counts().to_dict(), "→ 7+ sayfalı öğrencilerde bir yazı iki sayfaya taşmış olabilir; hangi yazı olduğu metin katmanından anlaşılamaz.")
print("Toplam sayfa - öğrenci x 6 =", int(per.sum()) - 6 * len(per))
# kodlar: sütunlar arası kayma var mı (aynı öğrenci blok sırası Excel sırasıyla aynı mı)
order_pdf = body.dropna(subset=["code"]).drop_duplicates("code").code.tolist()
print("PDF sırası Excel sırasıyla aynı mı:", order_pdf == [c for c in meta.Code if c in set(order_pdf)])
xl_order = {c: i for i, c in enumerate(meta.Code)}; pos_pdf = [xl_order[c] for c in order_pdf]
print("PDF sırası ile Excel satır sırası Spearman:", round(stats.spearmanr(range(len(pos_pdf)), pos_pdf)[0], 3), "| Excel sırasını bozan geçiş sayısı:", int((np.diff(pos_pdf) < 0).sum()))
spl = pdf_per_code[pdf_per_code > 1].index
for c in spl:
    g = body[body.code == c]; print(" PDF'lere bölünen öğrenci:", c, g.groupby("pdf").seite.agg(["min", "max", "count"]).reset_index().values.tolist())
print("Bu öğrencilerin sayfaları PDF sınırında mı (bir PDF'in sonu + sonrakinin başı):",
      all(body[body.code == c].groupby("pdf").seite.agg(lambda s: s.max() >= 700 or s.min() <= 3).all() for c in spl))
print("Ham metin kodu Excel'de olmayan sayfa:", body[body.code_text.notna() & ~body.code_text.isin(codes)][["pdf", "seite", "code_text", "code_neu", "code"]].to_dict("records"))

### Birebir kopya sayfalar (görüntü akışı SHA-1) ve yaklaşık kopya (küçük resim korelasyonu)

In [ ]:
bb = body[body.n_img > 0]
dup = bb[bb.img_sha1.duplicated(keep=False)].sort_values("img_sha1")
print("Birebir aynı görüntü akışı olan sayfa:", len(dup))
print(dup[["pdf", "seite", "code", "img_w", "img_h", "img_bytes"]].to_string())
print("\nÇok görüntülü sayfalar:", body[body.n_img > 1][["pdf", "seite", "code", "n_img"]].to_dict("records"))

# yaklaşık kopya: aynı (w,h) içinde normalize korelasyon
keys = [(r.pdf, r.seite) for r in body.itertuples() if (r.pdf, r.seite) in thumbs]
dims = {(r.pdf, r.seite): (r.img_w, r.img_h) for r in body.itertuples()}
codemap = {(r.pdf, r.seite): r.code for r in body.itertuples()}
Xt = np.stack([thumbs[k].ravel() for k in keys]); Xt = (Xt - Xt.mean(1, keepdims=True)) / (Xt.std(1, keepdims=True) + 1e-6)
C = Xt @ Xt.T / Xt.shape[1]; np.fill_diagonal(C, -1)
same_dim = np.array([[dims[a] == dims[b] for b in keys] for a in keys]); C[~same_dim] = -1
iu = np.triu_indices(len(keys), 1); vals = C[iu]
order = np.argsort(-vals)[:12]
top = pd.DataFrame([dict(a=keys[iu[0][o]], b=keys[iu[1][o]], r=round(float(vals[o]), 4), ayni_ogrenci=codemap[keys[iu[0][o]]] == codemap[keys[iu[1][o]]]) for o in order])
print("\nEn benzer sayfa çiftleri (küçük resim korelasyonu):"); print(top.to_string())
print("Korelasyon > 0.98 olan çift:", int((vals > 0.98).sum()), "| > 0.95:", int((vals > 0.95).sum()), "| > 0.90:", int((vals > 0.90).sum()))

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for a, o in zip(axs.ravel(), order[:4]):
    pass
sel = [(keys[iu[0][o]], keys[iu[1][o]]) for o in order[:4]]
for c, (ka, kb) in enumerate(sel):
    for r_, k in enumerate([ka, kb]):
        pm = pymupdf.open(RAW / k[0])[k[1] - 1].get_pixmap(dpi=40); im = Image.frombytes("RGB", (pm.width, pm.height), pm.samples)
        axs[r_, c].imshow(im); axs[r_, c].set_title(f"{k[0][-6:-4]} s{k[1]} {codemap[k]}", fontsize=8); axs[r_, c].axis("off")
plt.tight_layout(); plt.show()

## 5. Metadata dağılımları (öğrenci düzeyi)

In [ ]:
for c in ["Klassenstufe", "Gruppe", "Schulform", "Geschlecht", "Sprachbiographie"]:
    print(c, meta[c].value_counts().to_dict())
print("Deutschnote:", meta.Deutschnote.value_counts(dropna=False).sort_index().to_dict())
print("\nYaş (ay) sınıfa göre:"); print(meta.groupby("Klassenstufe")["Alter in Monaten"].describe().round(1))
print("\nSınıf x okul türü x grup (öğrenci sayısı):"); print(pd.crosstab([meta.Klassenstufe, meta.Schulform], meta.Gruppe))
print("\nSınıf x cinsiyet:"); print(pd.crosstab(meta.Klassenstufe, meta.Geschlecht))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for k, g in meta.groupby("Klassenstufe"): ax[0].hist(g["Alter in Monaten"].dropna(), bins=25, alpha=.6, label=k)
ax[0].legend(); ax[0].set_title("yaş (ay)")
meta.groupby(["Klassenstufe", "Gruppe"]).size().unstack().plot.bar(ax=ax[1]); ax[1].set_title("öğrenci sayısı"); plt.tight_layout(); plt.show()

### Kod yapısı: kod sınıf/okul bilgisi taşıyor mu? (hipotez, kontrol)

In [ ]:
meta["p1"] = meta.Code.str[:1]; meta["p2"] = meta.Code.str[1:2]; meta["p3"] = meta.Code.str[2:4]; meta["p4"] = meta.Code.str[4:6]
print("Kodun 1. karakteri x sınıf:"); print(pd.crosstab(meta.p1, meta.Klassenstufe))
print("2. karakter (K/H) x okul türü x sınıf:"); print(pd.crosstab(meta.p2, [meta.Klassenstufe, meta.Schulform]))
print("3-4. karakter (ör. 51, 52, 55...) x sınıf x grup:"); print(pd.crosstab(meta.p3, [meta.Klassenstufe, meta.Gruppe]))
print("Kodun ilk 4 karakteri (sınıf/okul benzeri küme) sayısı:", meta.Code.str[:4].nunique(),
      "| küme başına öğrenci medyan:", int(meta.Code.str[:4].value_counts().median()), "(anlam yalnızca desen olarak gözlendi, kod kitabı yok)")

## 6. Yazma görevi ve ölçüm zamanı
Excel sütunları: `Ber`/`Arg` (Bericht = bildirim, Argumentation = tartışma yazısı) x `MZP1..3` (üç ölçüm zamanı).
Sayfa → (görev, zaman) eşlemesi PDF'te yazmıyor. Hipotez: öğrencinin 6 sayfası sırayla Ber1, Arg1, Ber2, Arg2, Ber3, Arg3.
Doğrulama: (a) başlık kırpıntıları gözle, (b) sayfadaki mürekkep miktarının Excel `Textlänge` ile ilişkisi (Aşama 2 örnekleminde, 6x6 matris).

In [ ]:
TASKS = [("Ber", 1), ("Arg", 1), ("Ber", 2), ("Arg", 2), ("Ber", 3), ("Arg", 3)]
tl_cols = [f"Textlänge_{t}_MZP{m}" for t, m in TASKS]; rt_cols = [f"Rating_{t}_MZP{m}" for t, m in TASKS]
print("Textlänge özet (kelime sayısı olduğu yayında yazıyor):"); print(meta[tl_cols].describe().round(1).T)
print("\nRating özet:"); print(meta[rt_cols].describe().round(2).T)
vals = pd.Series(meta[rt_cols].values.ravel()).dropna()
print("\nRating değerleri:", sorted(vals.unique().tolist()), "| 0,5 adımlı mı:", bool(np.all((vals * 2) % 1 == 0)))
print("Rating-Textlänge Spearman (görev/zaman başına):", {c.split('_', 1)[1]: round(stats.spearmanr(meta[a], meta[c], nan_policy='omit')[0], 2) for a, c in zip(tl_cols, rt_cols)})
print("Rating sütunlarının ilişkisi: Ber1-Ber2 %.2f, Ber1-Arg1 %.2f" % (meta[rt_cols[0]].corr(meta[rt_cols[2]], method="spearman"), meta[rt_cols[0]].corr(meta[rt_cols[1]], method="spearman")))
print("Diğer sütunlar (anlamı belgede yok): AlphabetTask", meta.AlphabetTask.describe()[["min", "max"]].tolist(), "SLS", meta.SLS.describe()[["min", "max"]].tolist(),
      "Wortschatz", meta.Wortschatz.describe()[["min", "max"]].tolist())
print("Textlänge MZP1→MZP3 ortalama (Gruppe'ye göre):"); print(meta.groupby("Gruppe")[tl_cols].mean().round(1).T)

### Görsel kontrol: 6 öğrenci x 6 sayfa başlık kırpıntıları (sayfa sırası hipotezi)

In [ ]:
rng = random.Random(SEED)
good = per[per == 6].index.tolist(); pick = rng.sample(good, 6)
fig, axs = plt.subplots(6, 6, figsize=(20, 9))
for r_, code in enumerate(pick):
    rws = body[body.code == code].sort_values(["pdf", "seite"])
    for c, rr in enumerate(rws.itertuples()):
        pg = pymupdf.open(RAW / rr.pdf)[rr.seite - 1]; rc = pg.rect
        pm = pg.get_pixmap(dpi=70, clip=pymupdf.Rect(0, 0, rc.width, rc.height * 0.17)); im = Image.frombytes("RGB", (pm.width, pm.height), pm.samples)
        axs[r_, c].imshow(im); axs[r_, c].set_title(f"{code} | sıra {c + 1} → {TASKS[c][0]}{TASKS[c][1]}", fontsize=7); axs[r_, c].axis("off")
plt.tight_layout(); plt.show()

## 7. Görüntü çözünürlüğü ve renk biçimi (sayfa başına gömülü görüntü)

In [ ]:
b = body[body.n_img > 0].copy()
b["portrait"] = b.page_h_pt > b.page_w_pt
# etkin DPI: görüntünün uzun kenarı / sayfanın uzun kenarı (inç)
b["dpi_eff"] = (b[["img_w", "img_h"]].max(1) / (b[["page_w_pt", "page_h_pt"]].max(1) / 72)).round()
b["dpi_grp"] = pd.cut(b.dpi_eff, [0, 120, 170, 230, 400, 1000], labels=["<120", "~150", "~200", "~300", ">400"])
print("Etkin DPI grupları:", b.dpi_grp.value_counts().sort_index().to_dict())
print("En sık (w,h):", b.groupby(["img_w", "img_h"]).size().sort_values(ascending=False).head(5).to_dict())
print("Renk uzayı / bit:", b.groupby(["cs", "bpc"]).size().to_dict())
print("Sayfa dönüşü (/Rotate):", b.rotation.value_counts().to_dict(), "| sayfa boyutu (A4 ≈ 595x842 pt):", b.groupby(["page_w_pt", "page_h_pt"]).size().sort_values(ascending=False).head(3).to_dict())
print("\nPDF x etkin DPI:"); print(pd.crosstab(b.pdf, b.dpi_grp))
print("\nPDF x renk uzayı:"); print(pd.crosstab(b.pdf, b.cs))
print("\nGörüntü bayt/sayfa medyan (PDF'e göre, KB):"); print((b.groupby("pdf").img_bytes.median() / 1e3).round())
bm = b.merge(meta[["Code", "Klassenstufe", "Schulform", "Gruppe", "Geschlecht"]], left_on="code", right_on="Code", how="left")
print("\nPDF x sınıf (sayfa):"); print(pd.crosstab(bm.pdf, bm.Klassenstufe))
print("PDF x grup:"); print(pd.crosstab(bm.pdf, bm.Gruppe))
print("Etkin DPI x sınıf:"); print(pd.crosstab(bm.dpi_grp, bm.Klassenstufe))
print("Renk uzayı x sınıf:"); print(pd.crosstab(bm.cs, bm.Klassenstufe))

## 8. Manifest (`manifests/pages.csv`)
Sayfa başına bir satır; kullanıcı adı / kapak metni yok. Öğrenci düzeyi alanlar Excel'den eklenir. Sayfa → görev/zaman sütunları **hipotezdir** (`gorev_zaman_varsayim`).

In [ ]:
body["ogrenci_sayfa_sayisi"] = body.code.map(per)
body["ogrenci_sayfa_no"] = body.groupby("code").cumcount() + 1
body["gorev_zaman_varsayim"] = [f"{TASKS[n - 1][0]}_MZP{TASKS[n - 1][1]}" if (c in per.index and per[c] == 6) else None
                               for c, n in zip(body.code, body.ogrenci_sayfa_no)]
out = body.merge(meta[["Code", "Klassenstufe", "Gruppe", "Schulform", "Alter in Monaten", "Geschlecht"]], left_on="code", right_on="Code", how="left").drop(columns=["Code"])
out["dpi_eff"] = (out[["img_w", "img_h"]].max(1) / (out[["page_w_pt", "page_h_pt"]].max(1) / 72)).round()
cols = ["pdf", "seite", "code", "code_yontem", "code_text", "code_neu", "ogrenci_sayfa_no", "ogrenci_sayfa_sayisi", "gorev_zaman_varsayim", "Klassenstufe", "Gruppe", "Schulform",
        "Alter in Monaten", "Geschlecht", "n_img", "img_w", "img_h", "bpc", "cs", "filter", "img_bytes", "img_sha1", "dpi_eff", "page_w_pt", "page_h_pt", "rotation", "n_xxx", "text_extra_lines"]
out[cols].to_csv(MANIFESTS / "pages.csv", index=False, encoding="utf-8")
print("Yazıldı:", (MANIFESTS / "pages.csv").relative_to(ROOT), out.shape, "| git'te izlenir mi: hayır (data/ ignore)")

# Aşama 2: ön fizibilite (örneklem)
Örneklem: her PDF'ten rastgele 6 öğrenci (yalnızca tam 6 sayfalı) = 48 öğrenci, 288 sayfa; seed 42. Sayfalar 100 dpi'de bellekte oluşturulur (`get_pixmap` sayfa dönüşünü uygular).

In [ ]:
rng = random.Random(SEED); sample_codes = []
for pdf_name, g in body[body.code.isin(good)].groupby("pdf"):
    sample_codes += rng.sample(sorted(g.code.unique()), 6)
samp = body[body.code.isin(sample_codes)].copy()
samp["pos"] = samp.groupby("code").cumcount()
print("Öğrenci:", len(sample_codes), "| sayfa:", len(samp))
print(meta[meta.Code.isin(sample_codes)].groupby(["Klassenstufe", "Schulform"]).size().to_dict())
DPI = 100; MM = 25.4 / DPI

def render(row):
    pg = pymupdf.open(RAW / row.pdf)[row.seite - 1]
    pm = pg.get_pixmap(dpi=DPI, colorspace=pymupdf.csRGB)
    return np.frombuffer(pm.samples, np.uint8).reshape(pm.height, pm.width, 3)

def norm_gray(rgb):
    g = rgb.astype(np.float32).mean(2)
    bg = ndi.maximum_filter(ndi.gaussian_filter(g, 1.5), size=21); bg = ndi.gaussian_filter(bg, 8)
    return np.clip(g / np.maximum(bg, 1) * 255, 0, 255)

def skew_angle(mask, rng_deg=np.arange(-3, 3.01, 0.25)):
    sm = mask[::2, ::2].astype(np.float32); best, bs = 0.0, -1
    for a in rng_deg:
        s = ndi.rotate(sm, a, reshape=False, order=0).sum(1).var()
        if s > bs: best, bs = a, s
    for a in np.arange(best - 0.25, best + 0.26, 0.05):
        s = ndi.rotate(sm, a, reshape=False, order=0).sum(1).var()
        if s > bs: best, bs = a, s
    return float(best)

def rule_rows(opened, W, min_cov=0.25):
    cov = opened.sum(1) / W; rows = np.where(cov > min_cov)[0]
    if len(rows) == 0: return []
    groups = np.split(rows, np.where(np.diff(rows) > 2)[0] + 1)
    return [float(np.mean(g)) for g in groups]

def measure_page(rgb):
    H, W, _ = rgb.shape; gn = norm_gray(rgb); mask = gn < 170
    ang = skew_angle(mask)
    mask_d = ndi.rotate(mask.astype(np.uint8), ang, reshape=False, order=0).astype(bool)
    rgb_d = np.stack([ndi.rotate(rgb[..., k], ang, reshape=False, order=1, cval=255) for k in range(3)], 2)
    opened = ndi.binary_opening(mask_d, structure=np.ones((1, int(0.25 * W))))
    rules = rule_rows(opened, W)
    r = dict(skew_deg=ang, n_rules=len(rules), H=H, W=W)
    # renk: mürekkep pikselleri (kural dışı), baskın kalem rengi
    sat = rgb_d.max(2).astype(int) - rgb_d.min(2).astype(int)
    kill = ndi.binary_dilation(opened, structure=np.ones((5, 1)))
    ink = mask_d & ~kill
    r["bg_mean"] = float(rgb[gn > 200].mean()) if (gn > 200).any() else np.nan
    if len(rules) >= 3:
        sp = float(np.median(np.diff(rules))); r["rule_spacing_mm"] = sp * MM
        top, bot = rules[0] - 1.3 * sp, rules[-1] + 0.9 * sp
        roi = np.zeros_like(ink); roi[max(0, int(top)):int(min(H, bot))] = True; roi[:, :int(0.04 * W)] = False; roi[:, int(0.97 * W):] = False
        ink = ink & roi
        csel = (sat > 40) & ink; r["colored_frac"] = float(csel.sum() / max(ink.sum(), 1))
        if csel.sum() > 50:
            from skimage.color import rgb2hsv
            hh = rgb2hsv(rgb_d[csel][None, :, :].astype(np.uint8))[0, :, 0]; r["blue_share"] = float(((hh > 0.5) & (hh < 0.75)).mean())
        lab, n = ndi.label(ink, structure=np.ones((3, 3))); sl = ndi.find_objects(lab); sz = ndi.sum(ink, lab, range(1, n + 1))
        keep = [s for s, z in zip(sl, sz) if z >= 15]
        hs = np.array([s[0].stop - s[0].start for s in keep]); ws_ = np.array([s[1].stop - s[1].start for s in keep])
        r["n_comp"] = len(keep); r["comp_h_mm"] = float(np.median(hs) * MM) if len(hs) else np.nan
        r["comp_h_iqr_ratio"] = float((np.percentile(hs, 75) - np.percentile(hs, 25)) / np.median(hs)) if len(hs) > 5 else np.nan
        r["ink_frac"] = float(ink.sum() / roi.sum())
        # satır bantları: kural i'nin üstündeki bant
        bounds = [rules[0] - sp] + rules; offs, slopes, used = [], [], 0
        for k in range(1, len(bounds)):
            a_, b_ = int(bounds[k - 1] + 0.25 * sp), int(bounds[k] + 0.25 * sp)
            band = ink[max(a_, 0):max(b_, 0)]
            if band.sum() < 60: continue
            used += 1
            ys, xs = np.nonzero(band); cy = (ys.mean() + max(a_, 0)) - bounds[k]
            offs.append(cy / sp)
            xs_bins = np.arange(0, W, 40); pts = []
            for x0 in xs_bins:
                m_ = (xs >= x0) & (xs < x0 + 40)
                if m_.sum() > 25: pts.append((x0 + 20, ys[m_].mean()))
            if len(pts) >= 5:
                px, py = np.array(pts).T; slopes.append(np.degrees(np.arctan(np.polyfit(px, py, 1)[0])))
        r["bands_used"] = used; r["bands_total"] = len(rules); r["used_frac"] = used / max(len(rules), 1)
        r["base_offset_mean"] = float(np.mean(offs)) if offs else np.nan; r["base_offset_sd"] = float(np.std(offs)) if len(offs) > 1 else np.nan
        r["line_slope_med"] = float(np.median(slopes)) if slopes else np.nan; r["line_slope_sd"] = float(np.std(slopes)) if len(slopes) > 1 else np.nan
        # kelime boşluğu: bant sütun doluluğundaki boşluklar, harf yüksekliğine göre
        gaps = []
        for k in range(1, len(bounds)):
            a_, b_ = int(bounds[k - 1] + 0.25 * sp), int(bounds[k] + 0.25 * sp); band = ink[max(a_, 0):max(b_, 0)]
            if band.sum() < 60: continue
            occ = np.where(band.any(0))[0]
            if len(occ) > 1:
                dif = np.diff(occ); gaps += list(dif[dif > 1] - 1)
        gaps = np.array(gaps); hmed = np.median(hs) if len(hs) else np.nan
        r["n_gaps"] = len(gaps); r["wide_gaps"] = int((gaps > 0.6 * hmed).sum()) if len(gaps) else 0
        r["gap_med_over_h"] = float(np.median(gaps) / hmed) if len(gaps) and hmed == hmed else np.nan
        xs_all = np.nonzero(ink.any(0))[0]
        r["left_margin_mm"] = float(xs_all.min() * MM) if len(xs_all) else np.nan; r["right_margin_mm"] = float((W - xs_all.max()) * MM) if len(xs_all) else np.nan
        r["first_rule_mm"] = rules[0] * MM; r["last_rule_mm"] = rules[-1] * MM
    return r

feats = []
for k, row in enumerate(samp.itertuples()):
    try: m = measure_page(render(row)); m["err"] = None
    except Exception as e: m = dict(err=str(e)[:80])
    m.update(pdf=row.pdf, seite=row.seite, code=row.code, pos=row.pos, cs=row.cs, rotation=row.rotation)
    feats.append(m)
F = pd.DataFrame(feats); print("Ölçülen:", len(F), "| hata:", int(F.err.notna().sum()), "| ≥3 kural bulunan sayfa:", int((F.n_rules >= 3).sum()))
print(F[["skew_deg", "n_rules", "rule_spacing_mm", "comp_h_mm", "comp_h_iqr_ratio", "used_frac", "base_offset_mean", "base_offset_sd", "line_slope_med", "line_slope_sd",
         "gap_med_over_h", "wide_gaps", "left_margin_mm", "ink_frac", "colored_frac", "blue_share", "bg_mean"]].describe().round(2).T)

### Sayfa → (görev, zaman) hipotezinin sınaması
Sayfanın mürekkep miktarı ile Excel `Textlänge` (kelime sayısı) arasındaki Spearman korelasyonu: 6 sayfa konumu x 6 Excel sütunu.
Hipotez doğruysa köşegen en yüksek olmalı.

In [ ]:
mt = meta.set_index("Code")
sub = F[(F.n_rules >= 3)].copy()
mat = pd.DataFrame(index=[f"sayfa{p + 1}" for p in range(6)], columns=[f"{t}{m}" for t, m in TASKS], dtype=float)
for p in range(6):
    g = sub[sub.pos == p]
    for j, c in enumerate(tl_cols):
        mat.iloc[p, j] = stats.spearmanr(g.ink_frac * g.n_comp, g.code.map(mt[c]), nan_policy="omit")[0]
print((mat).round(2)); print("Her satırda en yüksek sütun:", mat.idxmax(1).tolist(), "| beklenen:", [f"{t}{m}" for t, m in TASKS])
print("Köşegen ortalama: %.2f | köşegen dışı ortalama: %.2f" % (np.mean(np.diag(mat.values)), (mat.values.sum() - np.trace(mat.values)) / 30))

## Ölçüm uygulanabilirliği: ölçüm başına sonuçlar (örneklem n=288 sayfa)

In [ ]:
Fm = F.merge(meta[["Code", "Klassenstufe", "Schulform", "Gruppe", "Geschlecht", "Alter in Monaten"]], left_on="code", right_on="Code")
Fm["task"] = Fm.pos.map(lambda p: TASKS[p][0]); Fm["mzp"] = Fm.pos.map(lambda p: TASKS[p][1])
print("Kural bulunamayan / < 3 kural sayfa oranı (renk uzayına göre):"); print(Fm.groupby("cs").apply(lambda g: round((g.n_rules.fillna(0) < 3).mean(), 2)))
print("\nSınıfa göre (medyan):"); print(Fm.groupby("Klassenstufe")[["rule_spacing_mm", "comp_h_mm", "comp_h_iqr_ratio", "used_frac", "line_slope_sd", "base_offset_sd", "left_margin_mm", "skew_deg"]].median().round(2).T)
print("\nÇizgi aralığı (mm) PDF'e göre:"); print(Fm.groupby("pdf").rule_spacing_mm.describe().round(1)[["count", "min", "50%", "max"]])
print("\nSayfa eğimi (tarama eğikliği, °) |açı| dağılımı:", Fm.skew_deg.abs().describe().round(2).to_dict())
print("Baskın kalem rengi: renkli mürekkep oranı >0.5 olan sayfa:", int((Fm.colored_frac > .5).sum()), "/", len(Fm), "| renk uzayına göre:", Fm.groupby("cs").colored_frac.median().round(2).to_dict())
fig, ax = plt.subplots(2, 3, figsize=(14, 6))
for a, c in zip(ax.ravel(), ["comp_h_mm", "rule_spacing_mm", "line_slope_sd", "used_frac", "left_margin_mm", "skew_deg"]):
    for k, g in Fm.groupby("Klassenstufe"): a.hist(g[c].dropna(), bins=20, alpha=.6, label=k)
    a.set_title(c); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

### Aynı öğrencinin farklı ölçüm zamanlarındaki değişim
Toplam varyansın ne kadarı öğrenciler arasında (kararlı özellik) ve ne kadarı aynı öğrencinin sayfaları arasında? ICC(1) tek yönlü ANOVA ile.

In [ ]:
def icc1(df, col):
    g = df.dropna(subset=[col]).groupby("code")[col].apply(list); g = g[g.apply(len) >= 2]
    k = g.apply(len).mean(); grand = np.concatenate(g.values).mean()
    msb = sum(len(v) * (np.mean(v) - grand) ** 2 for v in g) / (len(g) - 1)
    msw = sum(((np.array(v) - np.mean(v)) ** 2).sum() for v in g) / (sum(len(v) for v in g) - len(g))
    return (msb - msw) / (msb + (k - 1) * msw), len(g)
for c in ["comp_h_mm", "comp_h_iqr_ratio", "line_slope_med", "line_slope_sd", "base_offset_sd", "gap_med_over_h", "used_frac"]:
    v, n = icc1(Fm[Fm.n_rules >= 3], c); print(f"{c:20s} ICC(1)={v:5.2f} (n öğrenci={n})")
print("\nMZP1 → MZP3 (Ber ve Arg ortalaması), öğrenci eşleşmeli değişim, comp_h_mm:")
piv = Fm[Fm.n_rules >= 3].groupby(["code", "mzp"]).comp_h_mm.mean().unstack()
print(piv.describe().round(2)); print("MZP3-MZP1 farkı: ortalama %.2f mm, SD %.2f mm" % ((piv[3] - piv[1]).mean(), (piv[3] - piv[1]).std()))

## Karıştırıcı faktörler: görsel kontrol (örnek sayfalar)
12 rastgele sayfa (bellekte) incelenir; sayısal oran iddiası yoktur. Çizgisiz / yanlış yönlü sayfalar ayrıca listelenir.

In [ ]:
rng = random.Random(SEED + 1); pick = rng.sample(list(samp.itertuples()), 12)
fig, axs = plt.subplots(3, 4, figsize=(16, 13))
for a, row in zip(axs.ravel(), pick):
    pg = pymupdf.open(RAW / row.pdf)[row.seite - 1]; pm = pg.get_pixmap(dpi=45); im = Image.frombytes("RGB", (pm.width, pm.height), pm.samples)
    a.imshow(im); a.set_title(f"{row.pdf[-6:-4]} s{row.seite} {row.cs} rot{row.rotation}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
print("Sayfa dönüşü (rotation) doğrulaması: 90 / 180 / 270 dönüşlü sayfalardan 2'şer örnek (get_pixmap dönüşü uygular, dik çıkmalı)")
fig, axs = plt.subplots(1, 6, figsize=(16, 5)); k = 0
for rot in [90, 180, 270]:
    for row in body[body.rotation == rot].sample(2, random_state=SEED).itertuples():
        pg = pymupdf.open(RAW / row.pdf)[row.seite - 1]; pm = pg.get_pixmap(dpi=35); im = Image.frombytes("RGB", (pm.width, pm.height), pm.samples)
        axs[k].imshow(im); axs[k].set_title(f"rot{rot} {row.pdf[-6:-4]} s{row.seite}", fontsize=8); axs[k].axis("off"); k += 1
plt.tight_layout(); plt.show()
print("Kural bulunamayan (<3) örnek sayfalar:", Fm[Fm.n_rules.fillna(0) < 3][["pdf", "seite", "cs", "n_rules"]].head(12).to_dict("records"))

## Referans grup (akran dağılımı) kurulabilirliği
Hücre başına yazı sayısı: sınıf x görev x ölçüm zamanı, ek kırılımlarla. Bunlar **popülasyon normu değil**: Köln ve Hannover'de 2013–2015'te toplanan, 2 okul türünü (Gymnasium, IGS) kapsayan,
müdahale çalışmasına katılan bir örneklem **[yayında]**.

In [ ]:
n_stu = meta.groupby("Klassenstufe").size()
print("Öğrenci / sınıf:", n_stu.to_dict(), "| her öğrenciden 6 yazı → sınıf x görev x zaman hücresi başına yazı = öğrenci sayısı")
print("\nSınıf x okul türü x cinsiyet hücreleri (öğrenci):"); print(meta.groupby(["Klassenstufe", "Schulform", "Geschlecht"]).size().unstack())
print("\nSınıf x okul türü x cinsiyet x grup (en küçük hücre):", int(meta.groupby(["Klassenstufe", "Schulform", "Geschlecht", "Gruppe"]).size().min()))
print("Yaş (ay) 10'ar aylık dilimler x sınıf:"); print(pd.crosstab(pd.cut(meta["Alter in Monaten"], [100, 120, 130, 140, 150, 160, 170, 180, 190, 210]), meta.Klassenstufe))
print("Müdahale grubu MZP1'de müdahaleden önce mi? (Excel'de yok: MZP1 yalnızca bu varsayımla 'referans' sayılabilir; varsayım doğrulanmadı)")
print("Okul türü x sınıf x PDF karışıklığı: PDF'ler öğrenci sırasına göre bölünmüş (bkz. bölüm 7), tarama biçimi sınıf ve PDF ile ilişkili.")